In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")

# Parse timestamps as UTC datetimes
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# 1) metric.csv: group by cmdb_id and kpi_name, compute count, min, max, mean, global P95 of value.
# Compute group stats using the full available series
metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count='count',
    min_value='min',
    max_value='max',
    mean_value='mean',
    p95_value=lambda x: x.quantile(0.95)
).reset_index()

# Sort by count descending and keep top 20
metric_summary_top20 = metric_grp.sort_values('count', ascending=False).head(20)

# Overall earliest and latest timestamp in metric.csv
metric_time_range = pd.Series({
    'earliest_timestamp_utc': metric_df['timestamp'].min(),
    'latest_timestamp_utc': metric_df['timestamp'].max()
})

# Round numeric columns for compact display
for col in ['min_value','max_value','mean_value','p95_value']:
    if col in metric_summary_top20:
        metric_summary_top20[col] = metric_summary_top20[col].round(6)

# 2) trace.csv: unique cmdb_id and unique trace_name (show at most 50), earliest/latest timestamp
trace_unique_cmdb = pd.DataFrame(trace_df['cmdb_id'].dropna().unique(), columns=['cmdb_id'])
trace_unique_names = pd.DataFrame(pd.Series(trace_df['trace_name'].dropna().unique()).head(50).reset_index(drop=True), columns=['trace_name'])
trace_time_range = pd.Series({
    'earliest_timestamp_utc': trace_df['timestamp'].min() if not trace_df.empty else pd.NaT,
    'latest_timestamp_utc': trace_df['timestamp'].max() if not trace_df.empty else pd.NaT
})

# 3) log.csv: unique cmdb_id and unique log_name (show at most 50), earliest/latest timestamp
log_unique_cmdb = pd.DataFrame(log_df['cmdb_id'].dropna().unique(), columns=['cmdb_id'])
log_unique_names = pd.DataFrame(pd.Series(log_df['log_name'].dropna().unique()).head(50).reset_index(drop=True), columns=['log_name'])
log_time_range = pd.Series({
    'earliest_timestamp_utc': log_df['timestamp'].min() if not log_df.empty else pd.NaT,
    'latest_timestamp_utc': log_df['timestamp'].max() if not log_df.empty else pd.NaT
})

# Display compact results (variables returned so IPython shows them)
metric_summary_top20, metric_time_range, trace_unique_cmdb, trace_unique_names, trace_time_range, log_unique_cmdb, log_unique_names, log_time_range

```
Out[1]:
```
summary = (
    "Summary:\n"
    "- Metrics: Top metric groups (each with 25 samples) are for services 'carts', 'carts-db', 'catalogue', and 'catalogue-db' with KPIs like cpu, mem, latency-50/90, socket, workload, and diskio. "
    "Notable values: carts.cpu max ~50.01 (mean ~23.39, P95 ~49.73); carts.socket max ~14.15 (P95 14.0); carts-db.diskio mean ~2.06e6 (P95 ~2.38e6); catalogue-db.mem ~4.307e8 (very stable). "
    "Metric timestamps range from 2024-01-18 16:30:00 UTC to 2024-01-18 16:54:00 UTC.\n"
    "- Traces: trace.csv is empty (no trace cmdb_id or trace_name values present).\n"
    "- Logs: log.csv contains multiple services (catalogue, front-end, orders, payment, queue-master, shipping, user, carts, carts-db, session-db, orders-db). "
    "Logged metrics names are 'log.error_count' and 'log.row_count'. Log timestamps are the same window: 2024-01-18 16:30:00 UTC to 2024-01-18 16:54:00 UTC."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count     min_value     max_value    mean_value     p95_value
0          carts         cpu     25  1.163247e+00  5.001042e+01  2.338719e+01  4.973383e+01
1          carts  latency-50     25  1.178500e-02  3.668600e-02  2.147200e-02  3.388900e-02
2          carts  latency-90     25  2.261400e-02  8.762800e-02  5.278100e-02  8.702200e-02
3          carts         mem     25  2.065449e+08  2.137197e+08  2.099897e+08  2.136186e+08
4          carts      socket     25  9.550000e+00  1.415000e+01  1.187800e+01  1.400000e+01
5          carts    workload     25  7.447767e+00  8.966683e+00  8.189751e+00  8.636193e+00
6       carts-db         cpu     25  2.606996e+00  3.950852e+00  3.364181e+00  3.906745e+00
7       carts-db      diskio     25  1.686816e+06  2.487226e+06  2.062824e+06  2.378910e+06
8       carts-db         mem     25  7.791821e+07  8.398008e+07  8.080011e+07  8.369148e+07
9       carts-db      socket     25  6.000000e+00  7.000000e+00  6.330667e+00  7.000000e+00
10     catalogue         cpu     25  1.379030e-01  1.707870e-01  1.555580e-01  1.675100e-01
11     catalogue  latency-50     25  3.005000e-03  3.048000e-03  3.025000e-03  3.046000e-03
12     catalogue  latency-90     25  4.609000e-03  4.687000e-03  4.646000e-03  4.682000e-03
13     catalogue         mem     25  6.311936e+06  6.429355e+06  6.362896e+06  6.403359e+06
14     catalogue      socket     25  6.000000e+00  6.000000e+00  6.000000e+00  6.000000e+00
15     catalogue    workload     25  3.812217e+00  4.348967e+00  4.095024e+00  4.307060e+00
16  catalogue-db         cpu     25  1.703360e-01  2.067980e-01  1.922820e-01  2.021270e-01
17  catalogue-db      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
18  catalogue-db         mem     25  4.307354e+08  4.307394e+08  4.307369e+08  4.307389e+08
19  catalogue-db      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00, earliest_timestamp_utc   2024-01-18 16:30:00+00:00
latest_timestamp_utc     2024-01-18 16:54:00+00:00
dtype: datetime64[ns, UTC], Empty DataFrame
Columns: [cmdb_id]
Index: [], Empty DataFrame
Columns: [trace_name]
Index: [], earliest_timestamp_utc   NaT
latest_timestamp_utc     NaT
dtype: datetime64[ns],          cmdb_id
0      catalogue
1      front-end
2         orders
3        payment
4   queue-master
5       shipping
6           user
7          carts
8       carts-db
9     session-db
10     orders-db,           log_name
0  log.error_count
1    log.row_count, earliest_timestamp_utc   2024-01-18 16:30:00+00:00
latest_timestamp_utc     2024-01-18 16:54:00+00:00
dtype: datetime64[ns, UTC])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse metric_df already loaded in the session.
# Define incident window in UTC
start_ts = pd.to_datetime("2024-01-18 16:27:59", utc=True)
end_ts = pd.to_datetime("2024-01-18 16:57:59", utc=True)

# 1) Filter metric.csv to the incident window
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 9) Compute global thresholds (P95 and P90) using the entire metric series BEFORE filtering
p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index(name='p95')
p90_df = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.90).reset_index(name='p90')

# Helper function to detect anomalies given a threshold column name and label
def find_anomalies(window_df, threshold_df, thresh_col_name, label):
    # Merge threshold into window data
    df = window_df.merge(threshold_df, on=['cmdb_id', 'kpi_name'], how='left')
    # Drop groups with no global threshold (should be rare)
    df = df.dropna(subset=[thresh_col_name])
    # Identify anomaly rows where value >= threshold
    mask = df['value'] >= df[thresh_col_name]
    df_anom = df[mask].copy()
    if df_anom.empty:
        return pd.DataFrame()  # no anomalies
    # Aggregate per group
    grouped = df_anom.groupby(['cmdb_id', 'kpi_name'])
    summary = grouped.agg(
        anomaly_count=('value', 'size'),
        earliest_anomaly_timestamp_utc=('timestamp', 'min'),
        max_anomaly_value=('value', 'max')
    ).reset_index()
    # Attach the global threshold for ratio calculation
    summary = summary.merge(threshold_df, on=['cmdb_id', 'kpi_name'], how='left')
    # Compute severity ratio
    summary['max_over_threshold_ratio'] = (summary['max_anomaly_value'] - summary[thresh_col_name]) / summary[thresh_col_name]
    # Prepare up to 5 sample rows per group (timestamp | value)
    def sample_str(subdf):
        sub = subdf.sort_values('timestamp').head(5)
        # ISO format timestamps with timezone, round values for compactness
        parts = [f"{ts.isoformat()}|{v:.6g}" for ts, v in zip(sub['timestamp'], sub['value'])]
        return "; ".join(parts)
    samples = grouped.apply(sample_str).reset_index(name='samples')
    summary = summary.merge(samples, on=['cmdb_id', 'kpi_name'], how='left')
    # Add label about which threshold used
    summary['threshold_used'] = label
    # Select and order columns compactly
    summary = summary[[
        'cmdb_id', 'kpi_name', 'threshold_used', 'anomaly_count',
        'earliest_anomaly_timestamp_utc', 'max_anomaly_value',
        thresh_col_name, 'max_over_threshold_ratio', 'samples'
    ]]
    # Sort as requested: anomaly_count desc, then ratio desc. Limit to top 20.
    summary = summary.sort_values(['anomaly_count', 'max_over_threshold_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)
    # Round numeric columns for compact display
    summary['max_anomaly_value'] = summary['max_anomaly_value'].round(6)
    summary[thresh_col_name] = summary[thresh_col_name].round(6)
    summary['max_over_threshold_ratio'] = summary['max_over_threshold_ratio'].round(6)
    return summary

# 2) Try with P95 first
anomalies_p95 = find_anomalies(metric_window, p95_df, 'p95', 'P95')

if anomalies_p95.empty:
    # 3) If none found with P95, try P90
    anomalies_p90 = find_anomalies(metric_window, p90_df, 'p90', 'P90')
    anomalies_result = anomalies_p90
else:
    anomalies_result = anomalies_p95

# Display compact result: anomalies_result (top groups with sample rows)
anomalies_result

```
Out[2]:
```
summary = (
    "Summary (P95 anomalies in incident window 2024-01-18 16:27:59–16:57:59 UTC):\n"
    "- Anomalies were detected using the global P95 thresholds (no fallback to P90 was needed). 20 (cmdb_id, kpi_name) groups had values >= P95 in the window.\n"
    "- Many services show their socket metrics exactly at the P95 (anomaly_count=25, max_over_p95_ratio=0.0), i.e. they consistently hit the P95 level across the window (earliest samples at 2024-01-18T16:30:00Z).\n"
    "- Notable exceedances above P95 (higher concern):\n"
    "  • shipping.cpu — 2 anomalies, earliest 2024-01-18T16:44:00Z, max 1.297064 (≈78.7% above P95 = 0.725978).\n"
    "  • queue-master.cpu — 2 anomalies, earliest 2024-01-18T16:32:00Z, max 1.493002 (≈77.1% above P95 = 0.842855).\n"
    "  • carts.socket — 3 anomalies, earliest 2024-01-18T16:45:00Z, max 14.15 (≈1.07% above P95 = 14.0).\n"
    "  • payment.workload — 2 anomalies, earliest 2024-01-18T16:30:00Z, max 2.6 (≈17.7% above P95 = 2.2093).\n"
    "- Interpretation: CPU spikes in 'shipping' and 'queue-master' are the most significant deviations relative to their historical P95 and merit investigation. Many socket metrics are simply at their P95 level (not exceeding it), so they are less likely to indicate exceptional incidents beyond expected peak behavior."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name threshold_used  anomaly_count earliest_anomaly_timestamp_utc  max_anomaly_value           p95  max_over_threshold_ratio                                            samples
0           catalogue    socket            P95             25      2024-01-18 16:30:00+00:00       6.000000e+00  6.000000e+00                  0.000000  2024-01-18T16:30:00+00:00|6; 2024-01-18T16:31:...
1        catalogue-db    socket            P95             25      2024-01-18 16:30:00+00:00       4.000000e+00  4.000000e+00                  0.000000  2024-01-18T16:30:00+00:00|4; 2024-01-18T16:31:...
2             payment    socket            P95             25      2024-01-18 16:30:00+00:00       5.000000e+00  5.000000e+00                  0.000000  2024-01-18T16:30:00+00:00|5; 2024-01-18T16:31:...
3        queue-master    socket            P95             25      2024-01-18 16:30:00+00:00       3.000000e+00  3.000000e+00                  0.000000  2024-01-18T16:30:00+00:00|3; 2024-01-18T16:31:...
4            rabbitmq    socket            P95             25      2024-01-18 16:30:00+00:00       1.100000e+01  1.100000e+01                  0.000000  2024-01-18T16:30:00+00:00|11; 2024-01-18T16:31...
5   rabbitmq-exporter       mem            P95             25      2024-01-18 16:30:00+00:00       4.575232e+06  4.575232e+06                  0.000000  2024-01-18T16:30:00+00:00|4.57523e+06; 2024-01...
6   rabbitmq-exporter    socket            P95             25      2024-01-18 16:30:00+00:00       1.000000e+00  1.000000e+00                  0.000000  2024-01-18T16:30:00+00:00|1; 2024-01-18T16:31:...
7          session-db    socket            P95             25      2024-01-18 16:30:00+00:00       3.000000e+00  3.000000e+00                  0.000000  2024-01-18T16:30:00+00:00|3; 2024-01-18T16:31:...
8             user-db    socket            P95             25      2024-01-18 16:30:00+00:00       8.000000e+00  8.000000e+00                  0.000000  2024-01-18T16:30:00+00:00|8; 2024-01-18T16:31:...
9        catalogue-db    diskio            P95             25      2024-01-18 16:30:00+00:00       0.000000e+00  0.000000e+00                       NaN  2024-01-18T16:30:00+00:00|0; 2024-01-18T16:31:...
10          front-end     error            P95             25      2024-01-18 16:30:00+00:00       0.000000e+00  0.000000e+00                       NaN  2024-01-18T16:30:00+00:00|0; 2024-01-18T16:31:...
11             orders     error            P95             25      2024-01-18 16:30:00+00:00       0.000000e+00  0.000000e+00                       NaN  2024-01-18T16:30:00+00:00|0; 2024-01-18T16:31:...
12           rabbitmq    diskio            P95             25      2024-01-18 16:30:00+00:00       0.000000e+00  0.000000e+00                       NaN  2024-01-18T16:30:00+00:00|0; 2024-01-18T16:31:...
13               user    socket            P95             20      2024-01-18 16:30:00+00:00       1.600000e+01  1.600000e+01                  0.000000  2024-01-18T16:30:00+00:00|16; 2024-01-18T16:31...
14          orders-db    socket            P95             13      2024-01-18 16:30:00+00:00       9.000000e+00  9.000000e+00                  0.000000  2024-01-18T16:30:00+00:00|9; 2024-01-18T16:31:...
15           carts-db    socket            P95              8      2024-01-18 16:47:00+00:00       7.000000e+00  7.000000e+00                  0.000000  2024-01-18T16:47:00+00:00|7; 2024-01-18T16:48:...
16              carts    socket            P95              3      2024-01-18 16:45:00+00:00       1.415000e+01  1.400000e+01                  0.010714  2024-01-18T16:45:00+00:00|14; 2024-01-18T16:51...
17           shipping       cpu            P95              2      2024-01-18 16:44:00+00:00       1.297064e+00  7.259780e-01                  0.786643  2024-01-18T16:44:00+00:00|0.72917; 2024-01-18T...
18       queue-master       cpu            P95              2      2024-01-18 16:32:00+00:00       1.493002e+00  8.428550e-01                  0.771363  2024-01-18T16:32:00+00:00|1.493; 2024-01-18T16...
19            payment  workload            P95              2      2024-01-18 16:30:00+00:00       2.600000e+00  2.209300e+00                  0.176843  2024-01-18T16:30:00+00:00|2.6; 2024-01-18T16:3...```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric_df and log_df from session; load error_logs.csv
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

error_df = pd.read_csv("error_logs.csv")
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# 1) Define target groups and compute global P95 using the full metric series (before window filter)
targets = [
    ("shipping", "cpu"),
    ("queue-master", "cpu"),
    ("payment", "workload"),
    ("carts", "socket")
]
targets_df = pd.DataFrame(targets, columns=['cmdb_id', 'kpi_name'])

# Compute global P95 for all groups once
global_p95_all = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index(name='global_p95')

# Merge to get p95 for targets (if missing, will be NaN)
targets_with_p95 = targets_df.merge(global_p95_all, on=['cmdb_id', 'kpi_name'], how='left')

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-18 16:27:59", utc=True)
end_ts = pd.to_datetime("2024-01-18 16:57:59", utc=True)

# Filter metric to window
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Helper to analyze each target group
summaries = []
for _, row in targets_with_p95.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    gp95 = row['global_p95']
    group_rows = metric_window[(metric_window['cmdb_id'] == cmdb) & (metric_window['kpi_name'] == kpi)].copy()
    # Find anomaly rows where value >= global_p95 (if gp95 is NaN, result will be empty)
    if pd.isna(gp95):
        anomalies = group_rows.iloc[0:0]  # empty
    else:
        anomalies = group_rows[group_rows['value'] >= gp95].sort_values('timestamp').copy()
    anomaly_count = len(anomalies)
    earliest_anom_ts = anomalies['timestamp'].min() if anomaly_count>0 else pd.NaT
    max_anom_val = anomalies['value'].max() if anomaly_count>0 else np.nan
    # ratio: handle zero gp95
    if anomaly_count>0 and gp95 and gp95 != 0:
        max_over_p95_ratio = (max_anom_val - gp95) / gp95
    elif anomaly_count>0 and gp95 == 0:
        max_over_p95_ratio = np.nan
    else:
        max_over_p95_ratio = np.nan

    # Prepare up to 10 sample rows as "timestamp_iso|value"
    if anomaly_count>0:
        samples_df = anomalies[['timestamp','value']].head(10)
        samples = "; ".join([f"{ts.isoformat()}|{v:.6g}" for ts, v in zip(samples_df['timestamp'], samples_df['value'])])
    else:
        samples = ""

    # Consecutive-minute runs: compute longest run length and timestamps
    longest_run_len = 0
    longest_run_start = pd.NaT
    longest_run_end = pd.NaT
    if anomaly_count>0:
        t = anomalies['timestamp'].sort_values().reset_index(drop=True)
        # differences in seconds
        diffs = t.diff().fillna(pd.Timedelta(seconds=0))
        # A new run starts where diff != 1 minute
        new_run = (diffs != pd.Timedelta(minutes=1))
        run_id = new_run.cumsum()
        runs = pd.DataFrame({'timestamp': t, 'run_id': run_id})
        run_lengths = runs.groupby('run_id').agg(start_ts=('timestamp','min'), end_ts=('timestamp','max'), length=('timestamp','size')).reset_index()
        # Find max length and earliest if multiple
        max_len = run_lengths['length'].max()
        max_runs = run_lengths[run_lengths['length'] == max_len].sort_values('start_ts')
        if not max_runs.empty:
            longest = max_runs.iloc[0]
            longest_run_len = int(longest['length'])
            longest_run_start = longest['start_ts']
            longest_run_end = longest['end_ts']

    summaries.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': gp95,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp_utc': earliest_anom_ts,
        'max_anomaly_value': max_anom_val,
        'max_over_p95_ratio': max_over_p95_ratio,
        'longest_consecutive_run_length': longest_run_len,
        'longest_run_start_utc': longest_run_start,
        'longest_run_end_utc': longest_run_end,
        'samples': samples
    })

metric_target_summary = pd.DataFrame(summaries)

# Round numeric columns for compact display
metric_target_summary['global_p95'] = metric_target_summary['global_p95'].round(6)
metric_target_summary['max_anomaly_value'] = metric_target_summary['max_anomaly_value'].round(6)
metric_target_summary['max_over_p95_ratio'] = metric_target_summary['max_over_p95_ratio'].round(6)

# Also prepare a compact table of the actual anomaly rows per group (limit 10 rows per group)
anomaly_rows_list = []
for _, r in metric_target_summary.iterrows():
    if r['anomaly_count'] > 0:
        cmdb = r['cmdb_id']; kpi = r['kpi_name']
        rows = metric_window[(metric_window['cmdb_id']==cmdb)&(metric_window['kpi_name']==kpi)&(metric_window['value']>=r['global_p95'])].sort_values('timestamp').head(10)
        # select columns timestamp and value
        rows_sel = rows[['timestamp','value']].copy()
        rows_sel['cmdb_id'] = cmdb
        rows_sel['kpi_name'] = kpi
        anomaly_rows_list.append(rows_sel)

if anomaly_rows_list:
    metric_anomaly_rows = pd.concat(anomaly_rows_list, ignore_index=True)
else:
    metric_anomaly_rows = pd.DataFrame(columns=['timestamp','value','cmdb_id','kpi_name'])

# 2) Logs: filter to the same incident window and focus on specified cmdb_ids
services = ["shipping","queue-master","payment","carts"]
log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()
log_window_target = log_window[log_window['cmdb_id'].isin(services)].copy()

# Aggregate per (cmdb_id, log_name): count, sum(value), earliest, latest timestamp
logs_agg = log_window_target.groupby(['cmdb_id','log_name']).agg(
    row_count=('value','size'),
    value_sum=('value','sum'),
    earliest_timestamp_utc=('timestamp','min'),
    latest_timestamp_utc=('timestamp','max')
).reset_index()

# Rows where log_name == 'log.error_count' and value > 0 for these services (up to 20)
log_errors_positive = log_window_target[(log_window_target['log_name']=='log.error_count') & (log_window_target['value']>0)].sort_values('timestamp').head(20)
log_errors_positive = log_errors_positive[['timestamp','cmdb_id','log_name','value']].reset_index(drop=True)

# 3) error_logs.csv: filter to window and services list, select columns
error_window = error_df[(error_df['timestamp'] >= start_ts) & (error_df['timestamp'] <= end_ts)].copy()
error_hits = error_window[error_window['cmdb_id'].isin(services)][['timestamp','cmdb_id','message']].sort_values('timestamp').head(20).reset_index(drop=True)

# Final compact outputs (variables returned for IPython display)
metric_target_summary, metric_anomaly_rows, logs_agg, log_errors_positive, error_hits

```
Out[3]:
```
Summary (computed P95 from full series; window 2024-01-18 16:27:59 → 2024-01-18 16:57:59 UTC)

1) Metrics (global P95 and anomalies >= global P95)
- shipping / cpu
  - global_p95 = 0.725978
  - anomaly_count = 2
  - earliest = 2024-01-18T16:44:00Z
  - max_anomaly_value = 1.297064 (max_over_p95_ratio ≈ 0.7866 → ~78.7% above P95)
  - longest consecutive run = 1 minute (no multi-minute streak)
  - sample anomaly rows (timestamp | value): 2024-01-18T16:44:00+00:00|0.72917; 2024-01-18T16:47:00+00:00|1.297064

- queue-master / cpu
  - global_p95 = 0.842855
  - anomaly_count = 2
  - earliest = 2024-01-18T16:32:00Z
  - max_anomaly_value = 1.493002 (max_over_p95_ratio ≈ 0.7714 → ~77.1% above P95)
  - longest consecutive run = 1 minute
  - sample anomaly rows: 2024-01-18T16:32:00+00:00|1.493; 2024-01-18T16:37:00+00:00|0.865678

- payment / workload
  - global_p95 = 2.209300
  - anomaly_count = 2
  - earliest = 2024-01-18T16:30:00Z
  - max_anomaly_value = 2.600000 (max_over_p95_ratio ≈ 0.1768 → ~17.7% above P95)
  - longest consecutive run = 2 minutes (2024-01-18T16:30:00 → 2024-01-18T16:31:00)
  - sample anomaly rows: 2024-01-18T16:30:00+00:00|2.6; 2024-01-18T16:31:00+00:00|2.217733

- carts / socket
  - global_p95 = 14.000000
  - anomaly_count = 3
  - earliest = 2024-01-18T16:45:00Z
  - max_anomaly_value = 14.150000 (max_over_p95_ratio ≈ 0.0107 → ~1.07% above P95)
  - longest consecutive run = 1 minute
  - sample anomaly rows: 2024-01-18T16:45:00+00:00|14; 2024-01-18T16:51:00+00:00|14.15; 2024-01-18T16:53:00+00:00|14

Interpretation of metric findings:
- The largest relative CPU deviations vs. historical P95 are in shipping.cpu and queue-master.cpu (~77–79% above P95) — these are the most notable spikes.
- payment.workload shows a short 2-minute consecutive exceedance (milder ~18% above P95).
- carts.socket hits/briefly exceeds P95 but only marginally (~1%), and occurrences are non-consecutive.

2) Logs (window filtered; services: shipping, queue-master, payment, carts)
- Aggregates per (cmdb_id, log_name) for the four services:
  - carts: log.error_count — row_count=20, value_sum=0, timestamps 2024-01-18T16:31:00 → 2024-01-18T16:53:00
           log.row_count   — row_count=20, value_sum=1251
  - payment: log.error_count — row_count=25, value_sum=0, timestamps 2024-01-18T16:30:00 → 2024-01-18T16:54:00
             log.row_count  — row_count=25, value_sum=2431
  - queue-master: log.error_count — row_count=25, value_sum=0, timestamps 2024-01-18T16:30:00 → 2024-01-18T16:54:00
                  log.row_count  — row_count=25, value_sum=13239
  - shipping: log.error_count — row_count=25, value_sum=0, timestamps 2024-01-18T16:30:00 → 2024-01-18T16:54:00
              log.row_count  — row_count=25, value_sum=1471

- Rows where log_name == 'log.error_count' and value > 0 for these services:
  - None found in the window (no positive error_count rows).

3) error_logs.csv (window & selected services)
- No error_log rows for shipping, queue-master, payment, or carts in the window (no matches).

Overall conclusion:
- The most significant metric anomalies in the incident window are CPU spikes in shipping and queue-master (large relative exceedance of global P95). payment.workload had a short consecutive exceedance. carts.socket exceeded P95 only marginally and non-consecutively.
- Logs and error logs for these four services show no positive error_count entries during the window.

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id  kpi_name  global_p95  anomaly_count earliest_anomaly_timestamp_utc  max_anomaly_value  max_over_p95_ratio  longest_consecutive_run_length     longest_run_start_utc       longest_run_end_utc                                            samples
0      shipping       cpu    0.725978              2      2024-01-18 16:44:00+00:00           1.297064            0.786643                               1 2024-01-18 16:44:00+00:00 2024-01-18 16:44:00+00:00  2024-01-18T16:44:00+00:00|0.72917; 2024-01-18T...
1  queue-master       cpu    0.842855              2      2024-01-18 16:32:00+00:00           1.493002            0.771363                               1 2024-01-18 16:32:00+00:00 2024-01-18 16:32:00+00:00  2024-01-18T16:32:00+00:00|1.493; 2024-01-18T16...
2       payment  workload    2.209300              2      2024-01-18 16:30:00+00:00           2.600000            0.176843                               2 2024-01-18 16:30:00+00:00 2024-01-18 16:31:00+00:00  2024-01-18T16:30:00+00:00|2.6; 2024-01-18T16:3...
3         carts    socket   14.000000              3      2024-01-18 16:45:00+00:00          14.150000            0.010714                               1 2024-01-18 16:45:00+00:00 2024-01-18 16:45:00+00:00  2024-01-18T16:45:00+00:00|14; 2024-01-18T16:51...,                   timestamp      value       cmdb_id  kpi_name
0 2024-01-18 16:44:00+00:00   0.729170      shipping       cpu
1 2024-01-18 16:47:00+00:00   1.297064      shipping       cpu
2 2024-01-18 16:32:00+00:00   1.493002  queue-master       cpu
3 2024-01-18 16:37:00+00:00   0.865678  queue-master       cpu
4 2024-01-18 16:30:00+00:00   2.600000       payment  workload
5 2024-01-18 16:31:00+00:00   2.217733       payment  workload
6 2024-01-18 16:45:00+00:00  14.000000         carts    socket
7 2024-01-18 16:51:00+00:00  14.150000         carts    socket
8 2024-01-18 16:53:00+00:00  14.000000         carts    socket,         cmdb_id         log_name  row_count  value_sum    earliest_timestamp_utc      latest_timestamp_utc
0         carts  log.error_count         20          0 2024-01-18 16:31:00+00:00 2024-01-18 16:53:00+00:00
1         carts    log.row_count         20       1251 2024-01-18 16:31:00+00:00 2024-01-18 16:53:00+00:00
2       payment  log.error_count         25          0 2024-01-18 16:30:00+00:00 2024-01-18 16:54:00+00:00
3       payment    log.row_count         25       2431 2024-01-18 16:30:00+00:00 2024-01-18 16:54:00+00:00
4  queue-master  log.error_count         25          0 2024-01-18 16:30:00+00:00 2024-01-18 16:54:00+00:00
5  queue-master    log.row_count         25      13239 2024-01-18 16:30:00+00:00 2024-01-18 16:54:00+00:00
6      shipping  log.error_count         25          0 2024-01-18 16:30:00+00:00 2024-01-18 16:54:00+00:00
7      shipping    log.row_count         25       1471 2024-01-18 16:30:00+00:00 2024-01-18 16:54:00+00:00, Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse metric_df and log_df from session; if not present, load and parse timestamps
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Target timestamps (UTC aware)
ts_list = [
    pd.to_datetime("2024-01-18 16:32:00", utc=True),
    pd.to_datetime("2024-01-18 16:44:00", utc=True)
]

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series BEFORE filtering
global_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index(name='global_p95')

# Select metric rows whose timestamp equals either target timestamp
metric_selected = metric_df[metric_df['timestamp'].isin(ts_list)].copy()

# Merge global_p95 into selected rows
metric_selected = metric_selected.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Determine is_over_p95 (True if value >= global_p95 and global_p95 is not NaN)
metric_selected['is_over_p95'] = False
mask = metric_selected['global_p95'].notna() & (metric_selected['value'] >= metric_selected['global_p95'])
metric_selected.loc[mask, 'is_over_p95'] = True

# Prepare output columns and sorting
metric_selected_table = metric_selected.loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'global_p95', 'is_over_p95']].copy()
metric_selected_table = metric_selected_table.sort_values(['timestamp', 'is_over_p95'], ascending=[True, False]).head(200)

# Round numeric columns for compact display
metric_selected_table['value'] = metric_selected_table['value'].round(6)
metric_selected_table['global_p95'] = metric_selected_table['global_p95'].round(6)

# 2) From log.csv: rows at those exact timestamps for specified log_names
log_names = ['log.row_count', 'log.error_count']
log_selected_table = log_df[log_df['timestamp'].isin(ts_list) & log_df['log_name'].isin(log_names)].copy()
log_selected_table = log_selected_table.loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']].sort_values(['timestamp','cmdb_id']).reset_index(drop=True)

# If no rows match, log_selected_table will be an empty DataFrame (explicitly returned)
metric_selected_table, log_selected_table

```
Out[4]:
```
Summary for the two timestamps (2024-01-18T16:32:00Z and 2024-01-18T16:44:00Z)

Overall counts
- Metric rows exactly at those two timestamps: 150 rows checked.
- Log rows (log.row_count or log.error_count) at those timestamps: 38 rows.

Metrics vs global P95 (computed from full series)
- At 16:32:00 UTC many metric series are at or above their global P95 (is_over_p95 = True). Examples that are at/above P95 at 16:32:
  - catalogue / socket = 6.0 (global_p95 = 6.0) → over P95
  - catalogue-db / diskio = 0.0 (global_p95 = 0.0) → over P95 (equal)
  - catalogue-db / mem = 430,739,400 (global_p95 ≈ 430,738,900) → over P95
  - catalogue-db / socket = 4.0 (global_p95 = 4.0) → over P95
  - front-end / error = 0 (global_p95 = 0) → over P95 (equal)
  (Many other series at 16:32 also meet or exceed their P95 — the table shows is_over_p95 True for many rows at that timestamp.)

- At 16:44:00 UTC the majority of metric series in the sample are below their global P95 (is_over_p95 = False). Examples at 16:44 that are below P95:
  - user / mem = 8,143,667 (global_p95 ≈ 8,165,185) → under P95
  - user / socket = 15 (global_p95 = 16) → under P95
  - user / workload = 18.55118 (global_p95 ≈ 19.35648) → under P95
  - user-db / diskio = 1,539,036 (global_p95 ≈ 1,574,203) → under P95
  - user-db / mem = 48,591,260 (global_p95 ≈ 50,900,830) → under P95

Logs at those timestamps
- Found 38 log rows at the two exact timestamps for log.row_count and log.error_count across services (examples: carts log.row_count=50 at 16:32; shipping log.row_count=61 at 16:44).
- No log.error_count > 0 was found at these timestamps for the checked rows (error_count values are 0).

If you want the exact table rows (full 150 metric rows or full 38 log rows) or a filtered list for particular services/components, tell me which subset to show (I can display .head(n) or the full selection up to your limit).

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp       cmdb_id  kpi_name         value    global_p95  is_over_p95
14  2024-01-18 16:32:00+00:00     catalogue    socket  6.000000e+00  6.000000e+00         True
17  2024-01-18 16:32:00+00:00  catalogue-db    diskio  0.000000e+00  0.000000e+00         True
18  2024-01-18 16:32:00+00:00  catalogue-db       mem  4.307394e+08  4.307389e+08         True
19  2024-01-18 16:32:00+00:00  catalogue-db    socket  4.000000e+00  4.000000e+00         True
21  2024-01-18 16:32:00+00:00     front-end     error  0.000000e+00  0.000000e+00         True
..                        ...           ...       ...           ...           ...          ...
143 2024-01-18 16:44:00+00:00          user       mem  8.143667e+06  8.165185e+06        False
144 2024-01-18 16:44:00+00:00          user    socket  1.500000e+01  1.600000e+01        False
145 2024-01-18 16:44:00+00:00          user  workload  1.855118e+01  1.935648e+01        False
147 2024-01-18 16:44:00+00:00       user-db    diskio  1.539036e+06  1.574203e+06        False
148 2024-01-18 16:44:00+00:00       user-db       mem  4.859126e+07  5.090083e+07        False

[150 rows x 6 columns],                    timestamp     cmdb_id         log_name  value
0  2024-01-18 16:32:00+00:00       carts  log.error_count      0
1  2024-01-18 16:32:00+00:00       carts    log.row_count     50
2  2024-01-18 16:32:00+00:00    carts-db  log.error_count      0
3  2024-01-18 16:32:00+00:00    carts-db    log.row_count      1
4  2024-01-18 16:32:00+00:00   catalogue  log.error_count      0
..                       ...         ...              ...    ...
33 2024-01-18 16:44:00+00:00  session-db    log.row_count      5
34 2024-01-18 16:44:00+00:00    shipping  log.error_count      0
35 2024-01-18 16:44:00+00:00    shipping    log.row_count     61
36 2024-01-18 16:44:00+00:00        user  log.error_count      0
37 2024-01-18 16:44:00+00:00        user    log.row_count    597

[38 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Ensure metric_df is loaded and timestamps are parsed (reuse if present)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Target service and incident window (UTC)
service = 'payment'
start_ts = pd.to_datetime("2024-01-18 16:27:59", utc=True)
end_ts = pd.to_datetime("2024-01-18 16:57:59", utc=True)

# 1) List all kpi_name values for payment
payment_kpis = metric_df[metric_df['cmdb_id'] == service]['kpi_name'].dropna().unique()
payment_kpis = pd.Series(payment_kpis, name='kpi_name').to_frame()

# 2) Compute global P95 for each payment kpi_name using the full series (no time filtering)
global_p95_all = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index(name='global_p95')
payment_p95 = global_p95_all[(global_p95_all['cmdb_id'] == service)].reset_index(drop=True)

# 3) Filter metric_df to the incident window
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Prepare per-kpi summary for payment
summaries = []
anomaly_rows_list = []

for kpi in payment_p95['kpi_name'].unique():
    gp95_row = payment_p95[payment_p95['kpi_name'] == kpi]
    if gp95_row.empty:
        gp95 = np.nan
    else:
        gp95 = float(gp95_row['global_p95'].iloc[0])
    # rows for this kpi in window
    group_rows = metric_window[(metric_window['cmdb_id'] == service) & (metric_window['kpi_name'] == kpi)].copy().sort_values('timestamp')
    # anomalies: value >= global_p95 (if gp95 is NaN, none)
    if pd.isna(gp95):
        anomalies = group_rows.iloc[0:0]
    else:
        anomalies = group_rows[group_rows['value'] >= gp95].copy()
    anomaly_count = len(anomalies)
    earliest_anom = anomalies['timestamp'].min() if anomaly_count>0 else pd.NaT
    max_anom_val = anomalies['value'].max() if anomaly_count>0 else np.nan
    if anomaly_count>0 and (not pd.isna(gp95)) and gp95 != 0:
        max_over_p95_ratio = (max_anom_val - gp95) / gp95
    elif anomaly_count>0 and gp95 == 0:
        max_over_p95_ratio = np.nan
    else:
        max_over_p95_ratio = np.nan

    # longest consecutive-minute run among anomaly timestamps
    longest_run_len = 0
    longest_start = pd.NaT
    longest_end = pd.NaT
    if anomaly_count>0:
        t = anomalies['timestamp'].sort_values().reset_index(drop=True)
        diffs = t.diff().fillna(pd.Timedelta(seconds=0))
        new_run = (diffs != pd.Timedelta(minutes=1))
        run_id = new_run.cumsum()
        runs = pd.DataFrame({'timestamp': t, 'run_id': run_id})
        run_lengths = runs.groupby('run_id').agg(start_ts=('timestamp','min'), end_ts=('timestamp','max'), length=('timestamp','size')).reset_index()
        max_len = int(run_lengths['length'].max())
        max_runs = run_lengths[run_lengths['length'] == max_len].sort_values('start_ts')
        if not max_runs.empty:
            chosen = max_runs.iloc[0]
            longest_run_len = int(chosen['length'])
            longest_start = chosen['start_ts']
            longest_end = chosen['end_ts']

    # samples (up to 10)
    if anomaly_count>0:
        samples_df = anomalies[['timestamp','value']].head(10).copy()
        samples = "; ".join([f"{ts.isoformat()}|{v:.6g}" for ts,v in zip(samples_df['timestamp'], samples_df['value'])])
        # also collect rows for a compact table
        rows_sel = anomalies[['timestamp','value']].head(10).copy()
        rows_sel['kpi_name'] = kpi
        anomaly_rows_list.append(rows_sel)
    else:
        samples = ""

    summaries.append({
        'cmdb_id': service,
        'kpi_name': kpi,
        'global_p95': gp95,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp_utc': earliest_anom,
        'max_anomaly_value': max_anom_val,
        'max_over_p95_ratio': max_over_p95_ratio,
        'longest_consecutive_run_length': longest_run_len,
        'longest_run_start_utc': longest_start,
        'longest_run_end_utc': longest_end,
        'samples': samples
    })

metric_payment_summary = pd.DataFrame(summaries)

# Round numeric columns for compact display
metric_payment_summary['global_p95'] = metric_payment_summary['global_p95'].round(6)
metric_payment_summary['max_anomaly_value'] = metric_payment_summary['max_anomaly_value'].round(6)
metric_payment_summary['max_over_p95_ratio'] = metric_payment_summary['max_over_p95_ratio'].round(6)

# Prepare compact anomaly rows table (concatenate per-kpi samples)
if anomaly_rows_list:
    metric_payment_anomaly_rows = pd.concat(anomaly_rows_list, ignore_index=True)
    # reorder columns for clarity
    metric_payment_anomaly_rows = metric_payment_anomaly_rows[['kpi_name','timestamp','value']].sort_values(['kpi_name','timestamp']).reset_index(drop=True)
    metric_payment_anomaly_rows['value'] = metric_payment_anomaly_rows['value'].round(6)
else:
    metric_payment_anomaly_rows = pd.DataFrame(columns=['kpi_name','timestamp','value'])

# Final compact outputs
payment_kpis, payment_p95[['kpi_name','global_p95']], metric_payment_summary, metric_payment_anomaly_rows

```
Out[5]:
```
Summary for cmdb_id = "payment" (global P95 computed from the full series; window = 2024-01-18 16:27:59 → 2024-01-18 16:57:59 UTC)

Payment KPIs available:
- cpu
- latency-50
- latency-90
- mem
- socket
- workload

Per-KPI results (global_p95, anomalies within window, and run info)

1) cpu
- global_p95 = 0.083177
- anomaly_count = 2
- earliest_anomaly_timestamp_utc = 2024-01-18T16:45:00Z
- max_anomaly_value = 0.084397
- max_over_p95_ratio ≈ 0.0147 (≈1.47% over P95)
- longest consecutive anomaly run = 1 minute (start/end: 2024-01-18T16:45:00Z) — anomalies are non-consecutive
- sample anomaly rows (timestamp | value): 2024-01-18T16:45:00Z | 0.083202; 2024-01-18T16:49:00Z | 0.084397

2) latency-50
- global_p95 = 0.002174
- anomaly_count = 2
- earliest_anomaly_timestamp_utc = 2024-01-18T16:53:00Z
- max_anomaly_value = 0.002212
- max_over_p95_ratio ≈ 0.0174 (≈1.74% over P95)
- longest consecutive anomaly run = 2 minutes (2024-01-18T16:53:00Z → 2024-01-18T16:54:00Z)
- sample rows: 2024-01-18T16:53:00Z | 0.00218117; 2024-01-18T16:54:00Z | 0.002212

3) latency-90
- global_p95 = 0.004435
- anomaly_count = 2
- earliest_anomaly_timestamp_utc = 2024-01-18T16:53:00Z
- max_anomaly_value = 0.004442
- max_over_p95_ratio ≈ 0.0016 (≈0.16% over P95)
- longest consecutive anomaly run = 2 minutes (2024-01-18T16:53:00Z → 2024-01-18T16:54:00Z)
- sample rows: 2024-01-18T16:53:00Z | 0.00443623; 2024-01-18T16:54:00Z | 0.004442

4) mem
- global_p95 = 4,546,000
- anomaly_count = 2
- earliest_anomaly_timestamp_utc = 2024-01-18T16:53:00Z
- max_anomaly_value = 4,546,560
- max_over_p95_ratio ≈ 0.000123 (≈0.0123% over P95)
- longest consecutive anomaly run = 2 minutes (2024-01-18T16:53:00Z → 2024-01-18T16:54:00Z)
- sample rows: 2024-01-18T16:53:00Z | 4,546,560; 2024-01-18T16:54:00Z | 4,546,560

5) socket
- global_p95 = 5.0
- anomaly_count = 25 (every minute in the window)
- earliest_anomaly_timestamp_utc = 2024-01-18T16:30:00Z
- max_anomaly_value = 5.0
- max_over_p95_ratio = 0.0 (values equal to P95)
- longest consecutive anomaly run = 25 minutes (2024-01-18T16:30:00Z → 2024-01-18T16:54:00Z) — continuous across the window
- sample rows (first 10 shown): 2024-01-18T16:30:00Z|5; 2024-01-18T16:31:00Z|5; 2024-01-18T16:32:00Z|5; 2024-01-18T16:33:00Z|5; 2024-01-18T16:34:00Z|5; 2024-01-18T16:35:00Z|5; 2024-01-18T16:36:00Z|5; 2024-01-18T16:37:00Z|5; 2024-01-18T16:38:00Z|5; 2024-01-18T16:39:00Z|5

6) workload
- global_p95 = 2.209300
- anomaly_count = 2
- earliest_anomaly_timestamp_utc = 2024-01-18T16:30:00Z
- max_anomaly_value = 2.6
- max_over_p95_ratio ≈ 0.1768 (≈17.68% over P95)
- longest consecutive anomaly run = 2 minutes (2024-01-18T16:30:00Z → 2024-01-18T16:31:00Z)
- sample rows: 2024-01-18T16:30:00Z | 2.6; 2024-01-18T16:31:00Z | 2.217733

Quick interpretation
- socket is at P95 for the whole window (25-minute continuous run) but equals P95 (no exceedance).
- workload has the largest relative exceedance among payment KPIs (~17.7% above P95) with a 2-minute consecutive run.
- cpu, latency, and mem show small, brief exceedances (mostly 1–2 minute runs) and relatively small ratios vs P95.

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name
0         cpu
1  latency-50
2  latency-90
3         mem
4      socket
5    workload,      kpi_name    global_p95
0         cpu  8.317715e-02
1  latency-50  2.173812e-03
2  latency-90  4.435348e-03
3         mem  4.546000e+06
4      socket  5.000000e+00
5    workload  2.209300e+00,    cmdb_id    kpi_name    global_p95  anomaly_count earliest_anomaly_timestamp_utc  max_anomaly_value  max_over_p95_ratio  longest_consecutive_run_length     longest_run_start_utc       longest_run_end_utc                                            samples
0  payment         cpu  8.317700e-02              2      2024-01-18 16:45:00+00:00       8.439700e-02            0.014671                               1 2024-01-18 16:45:00+00:00 2024-01-18 16:45:00+00:00  2024-01-18T16:45:00+00:00|0.0832019; 2024-01-1...
1  payment  latency-50  2.174000e-03              2      2024-01-18 16:53:00+00:00       2.212000e-03            0.017373                               2 2024-01-18 16:53:00+00:00 2024-01-18 16:54:00+00:00  2024-01-18T16:53:00+00:00|0.00218117; 2024-01-...
2  payment  latency-90  4.435000e-03              2      2024-01-18 16:53:00+00:00       4.442000e-03            0.001571                               2 2024-01-18 16:53:00+00:00 2024-01-18 16:54:00+00:00  2024-01-18T16:53:00+00:00|0.00443623; 2024-01-...
3  payment         mem  4.546000e+06              2      2024-01-18 16:53:00+00:00       4.546560e+06            0.000123                               2 2024-01-18 16:53:00+00:00 2024-01-18 16:54:00+00:00  2024-01-18T16:53:00+00:00|4.54656e+06; 2024-01...
4  payment      socket  5.000000e+00             25      2024-01-18 16:30:00+00:00       5.000000e+00            0.000000                              25 2024-01-18 16:30:00+00:00 2024-01-18 16:54:00+00:00  2024-01-18T16:30:00+00:00|5; 2024-01-18T16:31:...
5  payment    workload  2.209300e+00              2      2024-01-18 16:30:00+00:00       2.600000e+00            0.176843                               2 2024-01-18 16:30:00+00:00 2024-01-18 16:31:00+00:00  2024-01-18T16:30:00+00:00|2.6; 2024-01-18T16:3...,       kpi_name                 timestamp         value
0          cpu 2024-01-18 16:45:00+00:00  8.320200e-02
1          cpu 2024-01-18 16:49:00+00:00  8.439700e-02
2   latency-50 2024-01-18 16:53:00+00:00  2.181000e-03
3   latency-50 2024-01-18 16:54:00+00:00  2.212000e-03
4   latency-90 2024-01-18 16:53:00+00:00  4.436000e-03
5   latency-90 2024-01-18 16:54:00+00:00  4.442000e-03
6          mem 2024-01-18 16:53:00+00:00  4.546560e+06
7          mem 2024-01-18 16:54:00+00:00  4.546560e+06
8       socket 2024-01-18 16:30:00+00:00  5.000000e+00
9       socket 2024-01-18 16:31:00+00:00  5.000000e+00
10      socket 2024-01-18 16:32:00+00:00  5.000000e+00
11      socket 2024-01-18 16:33:00+00:00  5.000000e+00
12      socket 2024-01-18 16:34:00+00:00  5.000000e+00
13      socket 2024-01-18 16:35:00+00:00  5.000000e+00
14      socket 2024-01-18 16:36:00+00:00  5.000000e+00
15      socket 2024-01-18 16:37:00+00:00  5.000000e+00
16      socket 2024-01-18 16:38:00+00:00  5.000000e+00
17      socket 2024-01-18 16:39:00+00:00  5.000000e+00
18    workload 2024-01-18 16:30:00+00:00  2.600000e+00
19    workload 2024-01-18 16:31:00+00:00  2.217733e+00)```
```